# Comparative Analysis of Numerical Methods for Black-Scholes PDE

This notebook performs a comprehensive comparative analysis of the implemented numerical methods (Analytical Black-Scholes, Binomial Tree, Monte Carlo, and Finite Difference) for option pricing. It loads data from the SQLite database, compares the results based on accuracy, computational efficiency, and provides visualizations of key findings.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
import os
import sys
import numpy as np

# Add the project root to sys.path for importing modules
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

from src.data import database
from src.options.base import Option
from src.analytical.black_scholes import BlackScholes

# --- Configuration ---DATABASE_PATH = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'option_pricing_data.db'))
print(f"Database path: {DATABASE_PATH}")

# --- Load Data ---conn = database.connect_db(DATABASE_PATH)

options_df = pd.read_sql_query("SELECT * FROM options", conn)
results_df = pd.read_sql_query("SELECT * FROM results", conn)
market_data_df = pd.read_sql_query("SELECT * FROM market_data", conn)

conn.close()

print("Data loaded successfully:")
print(f"Options: {len(options_df)} records")
print(f"Results: {len(results_df)} records")
print(f"Market Data: {len(market_data_df)} records")

# Merge options and results for easier analysis
merged_df = pd.merge(results_df, options_df, left_on='option_id', right_on='id', suffixes=('_result', '_option'))
merged_df.rename(columns={'id_option': 'option_params_id'}, inplace=True)

print("\nMerged DataFrame head:")
merged_df.head()

## 1. Price Comparison: Analytical vs. Numerical Methods (Single Run)

This section compares the option prices calculated by different methods for a single, selected option. We expect numerical methods to approximate the analytical Black-Scholes price for European options.

In [ ]:
# Filter for a specific option (e.g., the first call option in the database)# We'll pick an option that was run with default parameters, not part of the convergence studysingle_run_options = merged_df[~merged_df['method'].str.contains('steps|paths|M|N')]example_option_id = single_run_options[single_run_options['option_type'] == 'call'].iloc[0]['option_id']example_option_data = merged_df[merged_df['option_id'] == example_option_id].copy()
plt.figure(figsize=(12, 7))sns.barplot(x='method', y='price', data=example_option_data)plt.title(f'Price Comparison for Option ID {example_option_id} (S={example_option_data["S"].iloc[0]}, K={example_option_data["K"].iloc[0]}, T={example_option_data["T"].iloc[0]}, r={example_option_data["r"].iloc[0]}, sigma={example_option_data["sigma"].iloc[0]})')plt.ylabel('Option Price')plt.xlabel('Pricing Method')plt.xticks(rotation=45, ha='right')plt.grid(axis='y', linestyle='--')plt.tight_layout()plt.show()

## 2. Error Analysis (vs. Analytical Black-Scholes) (Single Run)

We analyze the absolute and relative errors of numerical methods compared to the Black-Scholes analytical solution for a single run. This helps in understanding the accuracy of each method.

In [ ]:
# Get analytical price for comparisonanalytical_price_row = example_option_data[example_option_data['method'] == 'BlackScholes']if not analytical_price_row.empty:    analytical_price = analytical_price_row['price'].iloc[0]        # Calculate absolute and relative error    example_option_data['absolute_error'] = abs(example_option_data['price'] - analytical_price)    example_option_data['relative_error'] = abs((example_option_data['price'] - analytical_price) / analytical_price)        # Filter out Black-Scholes itself for error plots    numerical_methods_errors = example_option_data[example_option_data['method'] != 'BlackScholes']    plt.figure(figsize=(14, 6))        plt.subplot(1, 2, 1)    sns.barplot(x='method', y='absolute_error', data=numerical_methods_errors)    plt.title(f'Absolute Error vs. Black-Scholes (Option ID {example_option_id})')    plt.ylabel('Absolute Error')    plt.xlabel('Numerical Method')    plt.xticks(rotation=45, ha='right')    plt.grid(axis='y', linestyle='--')    plt.subplot(1, 2, 2)    sns.barplot(x='method', y='relative_error', data=numerical_methods_errors)    plt.title(f'Relative Error vs. Black-Scholes (Option ID {example_option_id})')    plt.ylabel('Relative Error')    plt.xlabel('Numerical Method')    plt.xticks(rotation=45, ha='right')    plt.grid(axis='y', linestyle='--')    plt.tight_layout()    plt.show()else:    print("Analytical Black-Scholes price not found for error analysis.")

## 3. Computational Time Comparison (Single Run)

This section compares the computational time taken by each pricing method for a single run. Efficiency is a critical factor, especially for real-time applications or large-scale simulations.

In [ ]:
plt.figure(figsize=(12, 7))sns.barplot(x='method', y='computation_time', data=example_option_data)plt.title(f'Computational Time Comparison for Option ID {example_option_id}')plt.ylabel('Time (seconds)')plt.xlabel('Pricing Method')plt.xticks(rotation=45, ha='right')plt.grid(axis='y', linestyle='--')plt.yscale('log') # Use log scale for better visualization if times vary widelyplt.tight_layout()plt.show()

## 4. Greeks Comparison (Single Run)

Greeks (Delta, Gamma, Vega, Theta, Rho) measure the sensitivity of the option price to changes in underlying parameters. We compare the analytical Greeks with those from numerical methods (where available or easily derivable).

In [ ]:
greeks_columns = ['delta', 'gamma', 'vega', 'theta', 'rho']greeks_data = example_option_data[['method'] + greeks_columns].set_index('method')
fig, axes = plt.subplots(nrows=len(greeks_columns), ncols=1, figsize=(12, 5 * len(greeks_columns)))fig.suptitle(f'Greeks Comparison for Option ID {example_option_id}', y=1.02, fontsize=16)
for i, greek in enumerate(greeks_columns):    sns.barplot(x=greeks_data.index, y=greeks_data[greek], ax=axes[i])    axes[i].set_title(f'{greek.capitalize()}')    axes[i].set_ylabel(greek.capitalize())    axes[i].set_xlabel('')    axes[i].tick_params(axis='x', rotation=45, ha='right')    axes[i].grid(axis='y', linestyle='--')
plt.tight_layout()plt.show()

## 5. Convergence Analysis

This section analyzes the convergence of numerical methods as their key parameters (number of steps/paths) increase. We expect the numerical prices to converge towards the analytical Black-Scholes price.

In [ ]:
# Filter for the option used in the convergence study (assuming it's the last one inserted)convergence_option_id = options_df.iloc[-1]['id']convergence_data = merged_df[merged_df['option_id'] == convergence_option_id].copy()
# Get the analytical price for this optionanalytical_price_conv = BlackScholes.price(Option(S=convergence_data['S'].iloc[0],                                                 K=convergence_data['K'].iloc[0],                                                 T=convergence_data['T'].iloc[0],                                                 r=convergence_data['r'].iloc[0],                                                 sigma=convergence_data['sigma'].iloc[0],                                                 option_type=convergence_data['option_type'].iloc[0]))['price']
convergence_data['absolute_error'] = abs(convergence_data['price'] - analytical_price_conv)convergence_data['relative_error'] = abs((convergence_data['price'] - analytical_price_conv) / analytical_price_conv)
# Extract parameter values from method namesdef extract_param(method_name):    if 'BinomialTree' in method_name:        return int(method_name.split('_')[-2]) # e.g., BinomialTree_European_50_steps    elif 'MonteCarlo' in method_name:        return int(method_name.split('_')[-2]) # e.g., MonteCarlo_standard_10000_paths    elif 'FiniteDifference' in method_name:        return int(method_name.split('_')[-1].replace('N', '')) # e.g., FiniteDifference_explicit_M50_N50    return np.nan
convergence_data['param_value'] = convergence_data['method'].apply(extract_param)
# Filter out analytical and non-convergence study methodsconvergence_data_filtered = convergence_data[convergence_data['method'] != 'BlackScholes'].dropna(subset=['param_value'])
plt.figure(figsize=(18, 10))
# Plot Price vs. Parameterplt.subplot(2, 2, 1)sns.lineplot(x='param_value', y='price', hue='method', data=convergence_data_filtered)plt.axhline(y=analytical_price_conv, color='r', linestyle='--', label='Analytical Price')plt.title('Price vs. Parameter Value (Convergence)')plt.ylabel('Option Price')plt.xlabel('Parameter Value (N_steps/N_paths/N)')plt.xscale('log')plt.legend()plt.grid(True, linestyle='--')
# Plot Absolute Error vs. Parameterplt.subplot(2, 2, 2)sns.lineplot(x='param_value', y='absolute_error', hue='method', data=convergence_data_filtered)plt.title('Absolute Error vs. Parameter Value')plt.ylabel('Absolute Error')plt.xlabel('Parameter Value (N_steps/N_paths/N)')plt.xscale('log')plt.yscale('log')plt.legend()plt.grid(True, linestyle='--')
# Plot Computational Time vs. Parameterplt.subplot(2, 2, 3)sns.lineplot(x='param_value', y='computation_time', hue='method', data=convergence_data_filtered)plt.title('Computational Time vs. Parameter Value')plt.ylabel('Time (seconds)')plt.xlabel('Parameter Value (N_steps/N_paths/N)')plt.xscale('log')plt.yscale('log')plt.legend()plt.grid(True, linestyle='--')
plt.tight_layout()plt.show()

## 6. Market Data Validation: Model Prices vs. Market Prices

This section compares the model-generated prices with actual market prices fetched from `yfinance`. We use the mid-price (average of bid and ask) as a proxy for the market price.

In [ ]:
# Select a market option to compare withif not market_data_df.empty:    # For simplicity, let's pick one market option that matches our test_option parameters as closely as possible    # In a real scenario, you'd match by S, K, T, option_type, and trade_date        # Find the analytical price for the test option    analytical_price_for_test_option = merged_df[(merged_df['option_id'] == example_option_id) &                                                  (merged_df['method'] == 'BlackScholes')]['price'].iloc[0]
    # Find a corresponding market option    # This is a simplified match; a robust comparison would involve more sophisticated matching logic    matched_market_option = market_data_df[        (market_data_df['strike_price'] == example_option_data['K'].iloc[0]) &        (market_data_df['option_type'] == example_option_data['option_type'].iloc[0])    ].sort_values(by='expiration_date').iloc[0] # Take the earliest expiring match
    market_mid_price = (matched_market_option['bid_price'] + matched_market_option['ask_price']) / 2    market_implied_vol = matched_market_option['implied_volatility']
    print(f"Analytical Black-Scholes Price: {analytical_price_for_test_option:.4f}")    print(f"Market Mid-Price (matched option): {market_mid_price:.4f}")    print(f"Difference (Model - Market): {analytical_price_for_test_option - market_mid_price:.4f}")    print(f"Market Implied Volatility: {market_implied_vol:.4f}")
    # Plotting comparison    comparison_prices = pd.DataFrame({        'Source': ['Analytical Black-Scholes', 'Market Mid-Price'],        'Price': [analytical_price_for_test_option, market_mid_price]    })
    plt.figure(figsize=(8, 5))    sns.barplot(x='Source', y='Price', data=comparison_prices)    plt.title('Model Price vs. Market Price')    plt.ylabel('Price')    plt.grid(axis='y', linestyle='--')    plt.tight_layout()    plt.show()
else:    print("No market data available for validation.")

## 7. Implied Volatility Smile/Skew from Market Data

The Black-Scholes model assumes constant volatility, but real markets exhibit a 'volatility smile' or 'skew'. This plot visualizes the implied volatility across different strike prices for a given expiration from market data.

In [ ]:
if not market_data_df.empty:    # Filter for a specific ticker and expiration for clarity    spy_market_data = market_data_df[market_data_df['ticker'] == 'SPY']    if not spy_market_data.empty:        latest_expiration = spy_market_data['expiration_date'].max()        spy_market_data_latest_exp = spy_market_data[spy_market_data['expiration_date'] == latest_expiration].copy()                # Clean up implied volatility data (remove inf/-inf or NaN)        spy_market_data_latest_exp = spy_market_data_latest_exp[np.isfinite(spy_market_data_latest_exp['implied_volatility'])]
        if not spy_market_data_latest_exp.empty:            plt.figure(figsize=(12, 7))            sns.scatterplot(x='strike_price', y='implied_volatility', hue='option_type', data=spy_market_data_latest_exp)            plt.title(f'Implied Volatility Smile/Skew for SPY (Exp: {latest_expiration})')            plt.xlabel('Strike Price')            plt.ylabel('Implied Volatility')            plt.grid(True, linestyle='--')            plt.tight_layout()            plt.show()        else:            print("No valid implied volatility data for SPY for the latest expiration.")    else:        print("No SPY market data found for implied volatility analysis.")else:    print("No market data found for implied volatility analysis.")

## 8. Conclusion and Further Work

This analysis provides a foundational comparison of various option pricing models, including a detailed look at their convergence properties. Key observations include:- **Accuracy:** Black-Scholes provides a benchmark for European options. Numerical methods approximate this, with varying degrees of accuracy depending on parameters and method choice. Convergence plots illustrate how accuracy improves with increased computational effort.- **Computational Efficiency:** Different methods exhibit distinct performance characteristics, which are crucial for practical applications. The trade-off between accuracy and speed is evident in the convergence study.- **Market Discrepancies:** Real market prices and implied volatilities often deviate from simple Black-Scholes assumptions, highlighting the need for more advanced models or adjustments.
**Further Work could include:**- **American Option Analysis:** Comparing American option pricing from Binomial Trees with other benchmarks.- **Advanced Models:** Implementing models that account for volatility smile/skew (e.g., local volatility, stochastic volatility models).- **Real-time Data Integration:** Connecting to live data feeds for continuous validation.- **NSE Data Integration:** Expanding data collection to include the Nairobi Securities Exchange for emerging market analysis.